# CUDA reproducibility check — mitigation OLL fragility

We found the out-of-lung-localization (OLL) metric differs by ~0.10 between a CUDA run (Colab) and a CPU run (local) on the **same** checkpoints — larger than the mitigation effect itself. This notebook quantifies that by running the eval on CUDA **twice** (run A and run B) so we can see (a) whether CUDA reproduces *itself*, and (b) how far CUDA sits from the deterministic CPU reference.

**Important:** this uses the SAME checkpoints Jonathan evaluated on CPU — they are already committed in the repo at `notebooks/checkpoints/control.pt` and `masked.pt`. Do NOT retrain, or the comparison is confounded (device vs. retraining).

**Before running:** `Runtime → Change runtime type → GPU`.

**Send back:** `cuda_repro.zip` (12 CSVs: runA + runB × {pretrained,control,masked} × {auroc,oll}).

### 1. Clone + install

In [ ]:
!git clone -b feat/mitigation https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib pyyaml

### 2. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

### 3. Mount Drive + unzip data\nThe checkpoints (`control.pt`, `masked.pt`) are already in the repo at `notebooks/checkpoints/` (committed), so you only need the **data zip** from Drive. Edit `DATA_ZIP` to its path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to your data zip's Drive path >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/

# checkpoints are already in the cloned repo (notebooks/checkpoints/)
import os
need = ['data/chexpert/PNG_valid', 'data/chexpert/metadata.csv',
        'notebooks/checkpoints/control.pt', 'notebooks/checkpoints/masked.pt']
missing = [p for p in need if not os.path.exists(p)]
assert not missing, 'Missing: ' + str(missing) + ' — check DATA_ZIP path / repo clone.'
print('all inputs present:', need)

### 4. Run A — CUDA eval, all 4 labels → `runA/`

In [ ]:
!python -m src.mitigation_eval --tag pretrained --num-labels 4 --device cuda --output-dir runA
!python -m src.mitigation_eval --tag control --checkpoint notebooks/checkpoints/control.pt --num-labels 4 --device cuda --output-dir runA
!python -m src.mitigation_eval --tag masked  --checkpoint notebooks/checkpoints/masked.pt  --num-labels 4 --device cuda --output-dir runA

### 5. Run B — identical CUDA eval → `runB/` (tests CUDA-vs-CUDA reproducibility)

In [ ]:
!python -m src.mitigation_eval --tag pretrained --num-labels 4 --device cuda --output-dir runB
!python -m src.mitigation_eval --tag control --checkpoint notebooks/checkpoints/control.pt --num-labels 4 --device cuda --output-dir runB
!python -m src.mitigation_eval --tag masked  --checkpoint notebooks/checkpoints/masked.pt  --num-labels 4 --device cuda --output-dir runB

### 6. Quick look — does CUDA reproduce itself (runA vs runB)?

In [ ]:
import pandas as pd
for tag in ['pretrained', 'control', 'masked']:
    a = pd.read_csv(f'runA/mitigation_{tag}_oll.csv').set_index('label')['oll_pos_mean']
    b = pd.read_csv(f'runB/mitigation_{tag}_oll.csv').set_index('label')['oll_pos_mean']
    print(f'\n=== {tag} OLL: runA vs runB ===')
    for l in a.index:
        print(f'  {l:18s} {a[l]:.4f}  {b[l]:.4f}  (diff {b[l]-a[l]:+.4f})')

### 7. Zip both runs + download → send to Jonathan

In [ ]:
!zip -qr /content/cuda_repro.zip runA runB && echo done
from google.colab import files
files.download('/content/cuda_repro.zip')